# Smart Parking Availability Predictor - Part 3: Best Model Export

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

**Academic Year:** 2026-27 | **Evaluation Rubric:** Parts 2, 3 & 4

This notebook demonstrates:
1. Loading and verifying the serialized Random Forest production model and scaler pipeline.
2. Single-pass evaluation on the unseen chronological test set (Dec 6 - Dec 19, 2016).
3. Validating the metadata schema and operational inputs consumed by the Streamlit application.

In [1]:
# Setup Cell: Add root to sys.path and import libraries
import os
import sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
print('Environment configured successfully!')

Environment configured successfully!


In [2]:
# Load serialized model and test set
import json
import joblib
from sklearn.metrics import classification_report, accuracy_score, f1_score

model_path = 'models/parking_model.joblib' if os.path.exists('models/parking_model.joblib') else '../models/parking_model.joblib'
meta_path = 'models/model_metadata.json' if os.path.exists('models/model_metadata.json') else '../models/model_metadata.json'

model = joblib.load(model_path)
with open(meta_path) as f:
    meta = json.load(f)

print('Loaded Model:', meta['model_name'])
print('Dataset:', meta['dataset'])
print('Features Expected:', meta['features'])
print('Target Classes:', meta['target_classes'])

Loaded Model: Random Forest Classifier
Dataset: UCI Parking Birmingham (30 Car Parks, Oct-Dec 2016)
Features Expected: ['Capacity', 'occupancy_pct', 'hour', 'minute', 'dayofweek', 'is_weekend', 'weekofyear', 'hour_sin', 'hour_cos', 'SystemCodeNumber_enc', 'lag_1', 'lag_2', 'lag_3', 'rolling_mean_3', 'rolling_std_3']
Target Classes: ['Available', 'Nearly Full', 'Full']


In [3]:
# Run end-to-end inference verification on Held-Out Test Set
test_path = 'data/processed/test.csv' if os.path.exists('data/processed/test.csv') else '../data/processed/test.csv'
test_df = pd.read_csv(test_path)
X_test = test_df[meta['features']]
y_test = test_df['target_class']

y_pred_enc = model.predict(X_test)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder().fit(meta['target_classes'])
y_pred = le.inverse_transform(y_pred_enc)

acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')
print(f'Held-Out Test Accuracy: {acc:.4f}')
print(f'Held-Out Test Macro F1: {macro_f1:.4f}')
print('\nDetailed Classification Report:\n', classification_report(y_test, y_pred))

Held-Out Test Accuracy: 0.9606
Held-Out Test Macro F1: 0.9564



Detailed Classification Report:
               precision    recall  f1-score   support

   Available       0.97      0.98      0.98      2485
        Full       0.95      0.94      0.95       985
 Nearly Full       0.95      0.94      0.94      1838

    accuracy                           0.96      5308
   macro avg       0.96      0.96      0.96      5308
weighted avg       0.96      0.96      0.96      5308

